# DeepMTL2R Baseline Runner (Kaggle)

Notebook ini menjalankan eksperimen baseline (Single-Task vs Multi-Task Vanilla) menggunakan fungsi dari `run_baseline.py`, **tanpa bergantung pada debug config YAML**.

Output utama:
- semua metrik per-fold dan agregasi mean±std
- `baseline_summary.json`
- plotting perbandingan metrik

In [1]:
!git clone https://github.com/jteo0/DeepMTL2R.git

Cloning into 'DeepMTL2R'...
remote: Enumerating objects: 392, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 392 (delta 10), reused 14 (delta 10), pack-reused 371 (from 1)
Receiving objects: 100% (392/392), 17.47 MiB | 24.10 MiB/s, done.
Resolving deltas: 100% (176/176), done.


In [4]:
# Setup: install requirements and make repo importable
# If Kaggle internet is disabled, skip clone and ensure repo is provided as a Dataset instead.
import os
# Install dependencies (no-op if already installed)
print('Installing requirements (may take a while)...')
try:
    __import__('subprocess').run(['pip','install','-r','/kaggle/working/DeepMTL2R/requirements.txt'], check=False)
except Exception as e:
    print('pip install -r failed or skipped:', e)
# Install in editable mode so local imports work
try:
    __import__('subprocess').run(['pip','install','-e','/kaggle/working/DeepMTL2R'], check=False)
except Exception as e:
    print('pip install -e failed or skipped:', e)
# If dataset exists in /kaggle/input, copy to working folder so code can read from project-relative path
INPUT_DS = '/kaggle/input/mslr-web10k/MSLR-WEB10K'
TARGET_DS = '/kaggle/working/DeepMTL2R/datasets/MSLR-WEB10K'
if os.path.exists(INPUT_DS):
    os.makedirs(TARGET_DS, exist_ok=True)
    print('Copying dataset from', INPUT_DS, 'to', TARGET_DS)
    __import__('shutil').copytree(INPUT_DS, TARGET_DS, dirs_exist_ok=True)
else:
    print('No dataset found at', INPUT_DS, '- ensure DATASET_BASE_PATH points to valid folder or upload as Dataset')
# Ensure project is importable
import sys
if '/kaggle/working/DeepMTL2R' not in sys.path:
    sys.path.insert(0, '/kaggle/working/DeepMTL2R')
print('PYTHONPATH updated; setup complete')

Installing requirements (may take a while)...


ERROR: Could not open requirements file: [Errno 2] No such file or directory: '/kaggle/working/DeepMTL2R/requirements.txt'


Obtaining file:///kaggle/working/DeepMTL2R
No dataset found at /kaggle/input/mslr-web10k/MSLR-WEB10K - ensure DATASET_BASE_PATH points to valid folder or upload as Dataset
PYTHONPATH updated; setup complete


ERROR: file:///kaggle/working/DeepMTL2R does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [ ]:
# Jika perlu (Kaggle environment baru), jalankan install dependency:
# !pip install -r /kaggle/working/DeepMTL2R/requirements.txt

In [1]:
import os
import gc
import json
import random
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

Torch: 2.10.0+cu128
CUDA available: True


In [ ]:
# ==============================
# Runtime Config (Notebook-owned)
# ==============================
# Ubah sesuai lokasi repo/dataset Anda di Kaggle.

PROJECT_ROOT = '/kaggle/working/DeepMTL2R'
EXAMPLE_DIR = os.path.join(PROJECT_ROOT, 'examples', 'MSLR-WEB10K')

# Dataset path contoh Kaggle: /kaggle/input/<dataset-name>/MSLR-WEB10K
DATASET_BASE_PATH = '/kaggle/input/mslr-web10k/MSLR-WEB10K'

# ===== Debug config ditulis di Notebook (bukan dari YAML) =====
DEBUG_MODE = False
DEBUG_RATIO = 1e-6   # set 0.0 untuk full data
FOLDS = [1, 2]

# Task config
TASK_INDICES_ST = [0]
TASK_INDICES_MT = [0, 131]
LABEL_INDICES = [131, 132, 133, 135]
REDUCTION_METHOD = 'mean'

# Output
OUTPUT_DIR = os.path.join(EXAMPLE_DIR, 'outputs', 'results')
CHECKPOINT_DIR = os.path.join(EXAMPLE_DIR, 'checkpoints')

# Config JSON model baseline
CONFIG_GATING = os.path.join(EXAMPLE_DIR, 'configs', 'config_gating.json')

for p in [PROJECT_ROOT, EXAMPLE_DIR, OUTPUT_DIR, CHECKPOINT_DIR]:
    if not os.path.exists(p) and p in [OUTPUT_DIR, CHECKPOINT_DIR]:
        os.makedirs(p, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('EXAMPLE_DIR :', EXAMPLE_DIR)
print('DATASET_BASE_PATH:', DATASET_BASE_PATH)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('DEBUG_MODE:', DEBUG_MODE, '| DEBUG_RATIO:', DEBUG_RATIO)

PROJECT_ROOT: /kaggle/working/DeepMTL2R
EXAMPLE_DIR : /kaggle/working/DeepMTL2R/examples/MSLR-WEB10K
DATASET_BASE_PATH: https://www.kaggle.com/datasets/engkhaledmo/mslr-10k
OUTPUT_DIR: /kaggle/working/DeepMTL2R/examples/MSLR-WEB10K/outputs/results
DEBUG_MODE: True | DEBUG_RATIO: 1e-06


In [ ]:
# Override training epochs for non-debug runs (Notebook-owned)
# This keeps the notebook independent from the JSON default of epochs=2.
import json
EPOCHS_OVERRIDE = 50
ORIG_CONFIG_PATH = os.path.join(EXAMPLE_DIR, 'configs', 'config_gating.json')
OVERRIDE_CONFIG_PATH = os.path.join(EXAMPLE_DIR, 'configs', 'config_gating_epochs50.json')

with open(ORIG_CONFIG_PATH, 'r', encoding='utf-8') as f:
    _cfg = json.load(f)

_cfg.setdefault('training', {})
_cfg['training']['epochs'] = EPOCHS_OVERRIDE

with open(OVERRIDE_CONFIG_PATH, 'w', encoding='utf-8') as f:
    json.dump(_cfg, f, indent=2)

CONFIG_GATING = OVERRIDE_CONFIG_PATH
print('Config override written to:', CONFIG_GATING)
print('Epochs override:', EPOCHS_OVERRIDE)

In [ ]:
import sys
import os
import importlib.util

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
if EXAMPLE_DIR not in sys.path:
    sys.path.insert(0, EXAMPLE_DIR)

RUN_BASELINE_PATH = os.path.join(EXAMPLE_DIR, 'run_baseline.py')
if os.path.exists(RUN_BASELINE_PATH):
    spec = importlib.util.spec_from_file_location('run_baseline', RUN_BASELINE_PATH)
    rb = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(rb)
else:
    import run_baseline as rb

# Override variabel global modul -> tidak bergantung debug config YAML
rb.DATASET_BASE_PATH = DATASET_BASE_PATH
rb.FOLDS = FOLDS
rb.REDUCTION_METHOD = REDUCTION_METHOD
rb.LABEL_INDICES = LABEL_INDICES
rb.OUTPUT_DIR = OUTPUT_DIR
rb.CHECKPOINT_DIR = CHECKPOINT_DIR
rb.CONFIG_GATING = CONFIG_GATING
rb.DEBUG = DEBUG_MODE

print('run_baseline loaded from:', RUN_BASELINE_PATH)
print('run_baseline globals overridden from notebook runtime config.')

ModuleNotFoundError: No module named 'run_baseline'

In [ ]:
def summarize_metrics(agg_dict):
    out = {}
    for metric_name, values in agg_dict.items():
        arr = np.array(values, dtype=float)
        out[metric_name] = {
            'mean': float(np.mean(arr)),
            'std': float(np.std(arr, ddof=1)) if len(arr) > 1 else 0.0,
            'per_fold': [float(x) for x in arr.tolist()]
        }
    return out

def metrics_summary_to_df(summary_dict, experiment_name):
    rows = []
    for m, v in summary_dict.items():
        rows.append({
            'experiment': experiment_name,
            'metric': m,
            'mean': v['mean'],
            'std': v['std'],
        })
    return pd.DataFrame(rows).sort_values('metric').reset_index(drop=True)

In [ ]:
# ==============================
# Run Baseline Experiments
# ==============================
all_ndcg30_st = []
all_ndcg30_mt = []
metrics_agg_st = defaultdict(list)
metrics_agg_mt = defaultdict(list)
params_st = []
params_mt = []
fold_details = []

for fold in FOLDS:
    print('\n' + '=' * 70)
    print(f'Processing Fold {fold}')
    print('=' * 70)

    dataset_path = os.path.join(DATASET_BASE_PATH, f'Fold{fold}')
    if not os.path.exists(dataset_path):
        raise FileNotFoundError(f'Dataset path not found: {dataset_path}')

    config = rb.Config.from_json(CONFIG_GATING)

    max_rows = None
    if DEBUG_MODE and DEBUG_RATIO > 0:
        estimated_total_rows = 30000000
        max_rows = max(1, int(estimated_total_rows * DEBUG_RATIO))

    train_ds, val_ds = rb.load_libsvm_dataset(
        dataset_path, config.data.slate_length, config.data.validation_ds_role, max_rows=max_rows
    )

    nf = train_ds.shape[-1] - len(LABEL_INDICES)
    train_dl, val_dl = rb.create_data_loaders(
        train_ds, val_ds, config.data.num_workers, config.data.batch_size
    )

    # Single-task
    res_st = rb.run_training(
        experiment_name=f'Single-Task-Fold{fold}',
        task_indices=TASK_INDICES_ST,
        moo_method='ls',
        task_weights_tensor=torch.tensor([1.0]),
        dataset_path=dataset_path,
        train_dl=train_dl,
        val_dl=val_dl,
        nf=nf
    )

    task_metrics_st = res_st.get('per_task_metrics', {}).get(0, {})
    ndcg30_st = float(task_metrics_st.get('ndcg_30', 0.0))
    all_ndcg30_st.append(ndcg30_st)
    for k, v in task_metrics_st.items():
        metrics_agg_st[k].append(float(v))
    params_st.append(res_st.get('num_params'))

    # Multi-task vanilla
    num_tasks = len(TASK_INDICES_MT)
    res_mt = rb.run_training(
        experiment_name=f'Multi-Task-Vanilla-Fold{fold}',
        task_indices=TASK_INDICES_MT,
        moo_method='ls',
        task_weights_tensor=[1.0 / num_tasks] * num_tasks,
        dataset_path=dataset_path,
        train_dl=train_dl,
        val_dl=val_dl,
        nf=nf
    )

    task_metrics_mt = res_mt.get('per_task_metrics', {}).get(0, {})
    ndcg30_mt = float(task_metrics_mt.get('ndcg_30', 0.0))
    all_ndcg30_mt.append(ndcg30_mt)
    for k, v in task_metrics_mt.items():
        metrics_agg_mt[k].append(float(v))
    params_mt.append(res_mt.get('num_params'))

    fold_details.append({
        'fold': fold,
        'single_task_metrics': task_metrics_st,
        'multi_task_metrics': task_metrics_mt,
        'single_task_num_params': res_st.get('num_params'),
        'multi_task_num_params': res_mt.get('num_params')
    })

    del train_ds, val_ds, train_dl, val_dl
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

avg_ndcg30_st = float(np.mean(all_ndcg30_st)) if all_ndcg30_st else 0.0
avg_ndcg30_mt = float(np.mean(all_ndcg30_mt)) if all_ndcg30_mt else 0.0
delta_m = ((avg_ndcg30_mt - avg_ndcg30_st) / avg_ndcg30_st) * 100 if avg_ndcg30_st > 0 else 0.0

summary_st = summarize_metrics(metrics_agg_st)
summary_mt = summarize_metrics(metrics_agg_mt)

summary_out = {
    'folds': FOLDS,
    'debug_mode': DEBUG_MODE,
    'debug_ratio': DEBUG_RATIO,
    'delta_m_percent': float(delta_m),
    'single_task': {
        'ndcg30_avg': avg_ndcg30_st,
        'ndcg30_folds': [float(x) for x in all_ndcg30_st],
        'params_per_fold': params_st,
        'metrics': summary_st,
    },
    'multi_task': {
        'ndcg30_avg': avg_ndcg30_mt,
        'ndcg30_folds': [float(x) for x in all_ndcg30_mt],
        'params_per_fold': params_mt,
        'metrics': summary_mt,
    },
    'fold_details': fold_details
}

baseline_dir = os.path.join(OUTPUT_DIR, 'baselines')
os.makedirs(baseline_dir, exist_ok=True)
summary_path = os.path.join(baseline_dir, 'baseline_summary.json')
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary_out, f, indent=2, default=float)

print('\nBaseline summary saved to:', summary_path)
print(f'Single-Task NDCG@30 avg: {avg_ndcg30_st:.6f}')
print(f'Multi-Task  NDCG@30 avg: {avg_ndcg30_mt:.6f}')
print(f'Delta m%%: {delta_m:+.4f}%')

In [ ]:
# Tampilkan semua metrik agregat (mean ± std)
df_st = metrics_summary_to_df(summary_st, 'single_task')
df_mt = metrics_summary_to_df(summary_mt, 'multi_task')
df_all = pd.concat([df_st, df_mt], axis=0, ignore_index=True)

pivot_mean = df_all.pivot(index='metric', columns='experiment', values='mean').sort_index()
pivot_std = df_all.pivot(index='metric', columns='experiment', values='std').sort_index()

display(df_all.sort_values(['metric', 'experiment']).reset_index(drop=True))
print('\nMean table:')
display(pivot_mean)
print('\nStd table:')
display(pivot_std)

In [ ]:
# Plot 1: Perbandingan mean ± std semua metrik
metrics = sorted(set(df_st['metric']).intersection(set(df_mt['metric'])))
x = np.arange(len(metrics))
width = 0.4

st_means = [summary_st[m]['mean'] for m in metrics]
st_stds = [summary_st[m]['std'] for m in metrics]
mt_means = [summary_mt[m]['mean'] for m in metrics]
mt_stds = [summary_mt[m]['std'] for m in metrics]

plt.figure(figsize=(14, 6))
plt.bar(x - width/2, st_means, width, yerr=st_stds, capsize=4, label='Single-Task')
plt.bar(x + width/2, mt_means, width, yerr=mt_stds, capsize=4, label='Multi-Task Vanilla')
plt.xticks(x, metrics, rotation=45, ha='right')
plt.ylabel('Metric Value')
plt.title('Baseline Metrics Comparison (mean ± std across folds)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Trend per fold untuk metrik NDCG@k (jika tersedia)
ndcg_metrics = sorted([m for m in metrics if m.startswith('ndcg_')], key=lambda x: int(x.split('_')[1]))

if ndcg_metrics:
    ncols = 3
    nrows = int(np.ceil(len(ndcg_metrics) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows), squeeze=False)

    for i, m in enumerate(ndcg_metrics):
        r, c = divmod(i, ncols)
        ax = axes[r][c]
        ax.plot(FOLDS, summary_st[m]['per_fold'], marker='o', label='Single-Task')
        ax.plot(FOLDS, summary_mt[m]['per_fold'], marker='s', label='Multi-Task')
        ax.set_title(m.upper())
        ax.set_xlabel('Fold')
        ax.set_ylabel('Score')
        ax.legend()

    # hide unused axes
    total_axes = nrows * ncols
    for j in range(len(ndcg_metrics), total_axes):
        r, c = divmod(j, ncols)
        axes[r][c].axis('off')

    fig.suptitle('NDCG@k by Fold', y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No NDCG metrics found to plot.')

In [ ]:
# Plot 3: Delta m% (NDCG@30)
plt.figure(figsize=(6, 4))
color = 'tab:green' if delta_m >= 0 else 'tab:red'
plt.bar(['Delta m%'], [delta_m], color=color)
plt.axhline(0, color='black', linewidth=1)
plt.ylabel('Percent (%)')
plt.title('Average Relative Improvement (Delta m%)')
plt.tight_layout()
plt.show()

## Catatan
- Notebook ini khusus baseline (Single-Task vs Multi-Task Vanilla).
- Metrik khusus arsitektur tetap dipisahkan sesuai desain:
  - Effective Dimensionality Efficiency: hanya Matryoshka
  - Gating Sparsity Ratio: hanya Feature Gating
- Jika Anda ingin, notebook ini bisa diperluas untuk menjalankan ketiga eksperimen (Baseline, Matryoshka, Gating) dalam satu pipeline plotting gabungan.